(r-comparisons-honest)=

# Sensitivity analysis in ModernDiD and R

This comparison checks whether {func}`~moderndid.honest_did` and R's
`HonestDiD` package return the same robust confidence intervals for the
Medicaid expansion event study in the
{ref}`sensitivity analysis example <example_honest_did>`. `HonestDiD` is the
authors' own implementation of
[Rambachan and Roth (2023)](https://doi.org/10.1093/restud/rdad018), the paper
that ModernDiD's sensitivity analysis follows. Both packages receive the same
event-study coefficients and the same covariance matrix. From those inputs the
page compares the original interval under exact parallel trends and the robust
intervals under smoothness and relative magnitudes restrictions, for every
method the two packages share.

The original interval, the conditional test, and the conditional FLCI hybrid
agree to rounding error. The page traces each of the two remaining gaps to its
source. R's fixed-length intervals sit up to 7.5e-5 from ModernDiD's because R
estimates a quantile of the folded normal distribution by simulation. Once R
uses the exact quantile instead, the two packages agree to within 5.4e-7.
Because each package simulates the critical value of the conditional least
favorable hybrid with its own random draws, that method's intervals can differ
by one step of the grid of candidate effects. With only two placebo estimates,
R's lower bound on $M$ also stops with an indexing error where ModernDiD returns
a value.

Since the R functions run inside this Python session through `rpy2`, they read
the exact arrays that ModernDiD builds. The section on
{ref}`running the comparisons <r-comparisons-running>` explains how to
install the R environment and rerun these cells.

:::{admonition} What the comparison holds fixed
:class: note

Every gap on this page comes from the sensitivity analysis rather than from
the event study. The {ref}`staggered comparison <r-comparisons-staggered>`
checks the estimator behind the event study against R's `did` package on the
minimum wage data.
:::

In [1]:
import numpy as np
import pandas as pd
import polars as pl
import rpy2.robjects as ro
from rpy2.robjects import numpy2ri
from rpy2.robjects.packages import importr

import moderndid as did

with ro.default_converter.context():
    importr("HonestDiD")

pd.set_option("display.precision", 8)

Error importing in API mode: ImportError("dlopen(/Users/jordandeklerk/moderndid/.venv/lib/python3.12/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib\n  Referenced from: <05EE76C0-33BF-3D4C-BC56-84005241BC88> /Users/jordandeklerk/moderndid/.venv/lib/python3.12/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file)")
Trying to import in ABI mode.


## The event study both packages read

{func}`~moderndid.load_ehec` loads the panel of the sensitivity analysis
example, 46 states observed in every year from 2008 through 2019. Of those
states, 30 expanded Medicaid under the Affordable Care Act between 2014 and
2019. The outcome `dins` is the share of low-income adults without children who
had health insurance. The example's event study compares each cohort with the
states that haven't expanded yet and measures every estimate from event
time −1, the base year under `base_period="universal"`. Its window keeps the
five years on either side of expansion.

In [2]:
# States that hadn't expanded by 2019 get a 0, the value att_gt reads as never treated.
data = did.load_ehec().with_columns(expansion_year=pl.col("yexp2").fill_null(0).cast(pl.Int64))

# The att_gt and aggte calls from the sensitivity analysis example.
result = did.att_gt(
    data,
    yname="dins",
    tname="year",
    idname="stfips",
    gname="expansion_year",
    control_group="notyettreated",
    base_period="universal",
    cband=False,
)
event_study = did.aggte(result, type="dynamic", min_e=-5, max_e=5)
print(event_study)

 Aggregate Treatment Effects (Event Study)

 Overall summary of ATT's based on event-study/dynamic aggregation:

┌────────┬────────────┬────────────────────────┐
│    ATT │ Std. Error │ [95% Conf. Interval]   │
├────────┼────────────┼────────────────────────┤
│ 0.0688 │     0.0080 │ [  0.0532,   0.0845] * │
└────────┴────────────┴────────────────────────┘


 Dynamic Effects:

┌────────────┬──────────┬────────────┬────────────────────────────┐
│ Event time │ Estimate │ Std. Error │ [95% Pointwise Conf. Band] │
├────────────┼──────────┼────────────┼────────────────────────────┤
│         -5 │  -0.0125 │     0.0065 │ [-0.0253,  0.0003]         │
│         -4 │  -0.0067 │     0.0052 │ [-0.0168,  0.0035]         │
│         -3 │  -0.0038 │     0.0047 │ [-0.0129,  0.0054]         │
│         -2 │  -0.0049 │     0.0040 │ [-0.0128,  0.0029]         │
│         -1 │   0.0000 │         NA │ NA                         │
│          0 │   0.0453 │     0.0060 │ [ 0.0335,  0.0571] *       │
│        

/Users/jordandeklerk/moderndid/moderndid/core/preprocess/builders.py:226: UserWarning: Be aware that there are some small groups in your dataset.
Check groups: 2019, 2015, 2016, 2017
  warnings.warn(warning_msg)
/Users/jordandeklerk/moderndid/moderndid/did/att_gt.py:422: UserWarning: Not returning pre-test Wald statistic due to singular covariance matrix
  warnings.warn(


Besides the base year at event time −1, the event study holds four placebo
estimates at event times −5 to −2 and six effects at 0 to 5. Since every
estimate is measured from the base year, {func}`~moderndid.honest_did` drops
that row and passes the ten remaining estimates to the sensitivity analysis.
For an event study estimated without `clustervars`, as this one is, their
covariance is the outer product of their influence functions over the 46
states divided by 46 squared. The next cell builds both inputs the same way and
hands them to R.

:::{admonition} HonestDiD's wrapper ignores clusters
:class: warning

For an event study estimated with `clustervars`, {func}`~moderndid.honest_did`
sums the influence functions within each cluster before it forms the
covariance. The `honest_did` method that `HonestDiD` provides for event
studies from R's `did` package never sums within clusters ([line 52 of
`R/honest_did.R`](https://github.com/cran/HonestDiD/blob/0.2.6/R/honest_did.R#L52)).
For a clustered event study, report ModernDiD's intervals, since R's wrapper
starts from a covariance that treats units in one cluster as independent.
:::

In [3]:
def honest_inputs(study):
    """Return the inputs that honest_did builds from an event study without clusters."""
    # honest_did leaves out the base year at event time -1, the reference for every estimate.
    keep = study.event_times != -1
    influence = study.influence_func[:, keep]
    return study.att_by_event[keep], influence.T @ influence / influence.shape[0] ** 2


betahat, sigma = honest_inputs(event_study)
with (ro.default_converter + numpy2ri.converter).context():
    ro.globalenv["betahat"] = betahat
    ro.globalenv["sigma"] = sigma

## The original interval and fixed-length intervals

We start with the example's own analysis of the effect in the year of
expansion, smoothness bounds $M$ from 0 to 0.03 with fixed-length confidence
intervals (FLCI). Besides one robust interval for each $M$,
{func}`~moderndid.honest_did` returns the original interval, the estimate's 95
percent interval under exact parallel trends. In R, `constructOriginalCS`
computes the original interval and `createSensitivityResults` the robust ones.

In [4]:
# The example's smoothness analysis of the effect in the year of expansion.
m_vec = [0.0, 0.005, 0.01, 0.015, 0.02, 0.025, 0.03]
smoothness = did.honest_did(
    event_study, event_time=0, sensitivity_type="smoothness", m_vec=m_vec, method="FLCI"
)
print(smoothness.robust_ci)

shape: (7, 5)
┌───────────┬──────────┬────────┬─────────┬───────┐
│ lb        ┆ ub       ┆ method ┆ delta   ┆ m     │
│ ---       ┆ ---      ┆ ---    ┆ ---     ┆ ---   │
│ f64       ┆ f64      ┆ str    ┆ str     ┆ f64   │
╞═══════════╪══════════╪════════╪═════════╪═══════╡
│ 0.029463  ┆ 0.054555 ┆ FLCI   ┆ DeltaSD ┆ 0.0   │
│ 0.022185  ┆ 0.058612 ┆ FLCI   ┆ DeltaSD ┆ 0.005 │
│ 0.01725   ┆ 0.063417 ┆ FLCI   ┆ DeltaSD ┆ 0.01  │
│ 0.012251  ┆ 0.068415 ┆ FLCI   ┆ DeltaSD ┆ 0.015 │
│ 0.007251  ┆ 0.073415 ┆ FLCI   ┆ DeltaSD ┆ 0.02  │
│ 0.002251  ┆ 0.078415 ┆ FLCI   ┆ DeltaSD ┆ 0.025 │
│ -0.002749 ┆ 0.083415 ┆ FLCI   ┆ DeltaSD ┆ 0.03  │
└───────────┴──────────┴────────┴─────────┴───────┘


In [5]:
# The first four coefficients are placebo estimates and the last six are effects.
with ro.default_converter.context():
    ro.globalenv["m_vec"] = ro.FloatVector(m_vec)
    ro.r("""
        r_original <- HonestDiD::constructOriginalCS(
            betahat, sigma, numPrePeriods = 4, numPostPeriods = 6
        )
        r_flci <- HonestDiD::createSensitivityResults(
            betahat, sigma, numPrePeriods = 4, numPostPeriods = 6, method = "FLCI", Mvec = m_vec
        )
    """)
    r_original = np.asarray(ro.r("c(r_original$lb, r_original$ub)"))
    r_flci = np.asarray(ro.r("cbind(r_flci$lb, r_flci$ub)"))

In [6]:
def side_by_side(labels, python_bounds, r_bounds):
    """Set each interval from ModernDiD beside the one from R and subtract R's ends."""
    table = labels.copy()
    table["ModernDiD lb"] = python_bounds[:, 0]
    table["R lb"] = r_bounds[:, 0]
    table["lb difference"] = table["ModernDiD lb"] - table["R lb"]
    table["ModernDiD ub"] = python_bounds[:, 1]
    table["R ub"] = r_bounds[:, 1]
    table["ub difference"] = table["ModernDiD ub"] - table["R ub"]
    return table


# The original interval assumes exact parallel trends and has no smoothness bound.
python_flci = np.vstack(
    [
        [smoothness.original_ci.lb, smoothness.original_ci.ub],
        smoothness.robust_ci.select("lb", "ub").to_numpy(),
    ]
)
flci_comparison = side_by_side(
    pd.DataFrame({"method": ["Original", *smoothness.robust_ci["method"]], "m": [np.nan, *m_vec]}),
    python_flci,
    np.vstack([r_original, r_flci]),
)
flci_comparison

,method,m,ModernDiD lb,R lb,lb difference,ModernDiD ub,R ub,ub difference
0,Original,NaN,0.03347873,0.03347873,-6.93889390e-18,0.05707169,0.05707169,6.93889390e-18
1,FLCI,0.000,0.02946346,0.02944944,1.40185236e-05,0.05455456,0.05456966,-1.51019004e-05
2,FLCI,0.005,0.02218509,0.02212935,5.57386314e-05,0.05861195,0.05853679,7.51653402e-05
3,FLCI,0.010,0.01724953,0.01726200,-1.24646111e-05,0.06341661,0.06340414,1.24704482e-05
4,FLCI,0.015,0.01225075,0.01226292,-1.21640165e-05,0.06841539,0.06840322,1.21698537e-05
5,FLCI,0.020,0.00725075,0.00726292,-1.21666032e-05,0.07341539,0.07340322,1.21724403e-05
6,FLCI,0.025,0.00225075,0.00226292,-1.21715275e-05,0.07841540,0.07840322,1.21773646e-05
7,FLCI,0.030,-0.00274926,-0.00273708,-1.21764528e-05,0.08341540,0.08340322,1.21822899e-05


The original interval matches to rounding error, since both packages form it
as the same estimate plus or minus the same normal quantile times the same
standard error. The fixed-length intervals differ by 1.2e-5 to 1.5e-5 at every
$M$ except 0.005. At that bound the gap grows to 7.5e-5 at the upper end.

(r-comparisons-honest-quantile)=

### A simulated quantile in R

Each fixed-length interval is a weighted sum of the ten estimates plus or minus
a half-length. That half-length is the weighted sum's standard deviation times
a quantile of the folded normal distribution, the distribution of $|Z + t|$ for
a standard normal $Z$ and the ratio $t$ of the worst-case bias to that standard
deviation. Where {func}`~moderndid.folded_normal_quantile` solves for this
quantile exactly, `HonestDiD` reads it off 1,000,000 simulated draws with seed 0
([lines 97 to 104 of `R/flci.R`](https://github.com/cran/HonestDiD/blob/0.2.6/R/flci.R#L97-L104)).
The next cell compares the two quantiles at zero bias and the interval lengths
at $M = 0$.

In [7]:
# R simulates the quantile at zero bias and ModernDiD solves for it exactly.
with ro.default_converter.context():
    simulated = ro.r("HonestDiD:::.qfoldednormal(p = 0.95, mu = 0)")[0]
exact = did.folded_normal_quantile(0.95, mu=0.0)

# At M = 0 the two packages choose estimators with the same standard deviation.
at_zero = flci_comparison[flci_comparison["m"] == 0].iloc[0]
r_length = at_zero["R ub"] - at_zero["R lb"]
python_length = at_zero["ModernDiD ub"] - at_zero["ModernDiD lb"]
print(f"quantile at zero bias: {simulated:.5f} in R, {exact:.5f} in ModernDiD")
print(f"ratio of the quantiles: {simulated / exact:.5f}")
print(f"ratio of the interval lengths at M = 0: {r_length / python_length:.5f}")

quantile at zero bias: 1.96224 in R, 1.95996 in ModernDiD
ratio of the quantiles: 1.00116
ratio of the interval lengths at M = 0: 1.00116


At zero bias the exact quantile is 1.95996, the 97.5th percentile of the
standard normal distribution. R's simulation gives 1.96224 instead, about 0.12
percent more than the exact value. Because both packages choose estimators
with the same standard deviation at $M = 0$, R's interval there is longer by
that same ratio. The cell below swaps R's simulated quantile for the exact one,
reruns `createSensitivityResults`, and puts R's own function back.

In [8]:
# Replace R's simulated quantile with the exact one, rerun the intervals, and restore R's function.
with ro.default_converter.context():
    ro.r("""
        simulated_quantile <- HonestDiD:::.qfoldednormal
        exact_quantile <- function(p, mu = 0, sd = 1, numSims = 10^6, seed = 0) {
            vapply(abs(mu) / sd, function(shift) {
                gap <- function(x) pnorm(x - shift) - pnorm(-x - shift) - p
                uniroot(gap, c(0, shift + 10), tol = 1e-14)$root
            }, numeric(1)) * sd
        }
        assignInNamespace(".qfoldednormal", exact_quantile, ns = "HonestDiD")
        r_flci_exact <- HonestDiD::createSensitivityResults(
            betahat, sigma, numPrePeriods = 4, numPostPeriods = 6, method = "FLCI", Mvec = m_vec
        )
        assignInNamespace(".qfoldednormal", simulated_quantile, ns = "HonestDiD")
    """)
    r_flci_exact = np.asarray(ro.r("cbind(r_flci_exact$lb, r_flci_exact$ub)"))

flci_exact_comparison = side_by_side(
    flci_comparison[["method", "m"]].iloc[1:].reset_index(drop=True),
    smoothness.robust_ci.select("lb", "ub").to_numpy(),
    r_flci_exact,
)
exact_gap = flci_exact_comparison[["lb difference", "ub difference"]].abs().to_numpy().max()
print(f"largest gap with the exact quantile in R: {exact_gap:.1e}")

largest gap with the exact quantile in R: 5.4e-07


With the exact quantile, R's fixed-length intervals match ModernDiD's to within
5.4e-7 at every $M$. The swap also closes the larger gap that the first table
shows at $M = 0.005$. With its simulated quantile, R's search for the shortest
interval at that bound stops at the least biased estimator. In exact terms that
estimator's interval is 1.2e-5 longer than the one ModernDiD finds. The largest
remaining gap sits at $M = 0$ and comes from R's solvers rather than the
quantile.

:::{admonition} Solver tolerance in R
:class: note

At $M = 0$ the shortest interval comes from the estimator with the smallest
variance, since every estimator either package considers is unbiased there.
ModernDiD's center matches the exact center of that estimator to within 2e-11.
R's solvers stop instead at nearby weights whose standard deviation matches
that estimator's to within 1e-10 and whose center sits 5.4e-7 away.
:::

## Test inversions on a shared grid

The conditional test (`"Conditional"`), the conditional FLCI hybrid (`"C-F"`),
and the conditional least favorable hybrid (`"C-LF"`) each check every
candidate effect on a grid. The interval a method reports runs from the
smallest candidate it accepts to the largest. To compare the tests rather than
the grids, we give both packages the same 301 candidates from −0.03 to 0.12,
one every 0.0005. Since `createSensitivityResults` takes no grid, the R side
calls `computeConditionalCS_DeltaSD` for each value of $M$ and
`computeConditionalCS_DeltaSDB` for the example's check with a positive bias.

:::{admonition} Pass the same grid to both packages
:class: tip

For the conditional test and the least favorable hybrid, ModernDiD centers its
default grid on the identified set that the estimates imply. `HonestDiD`
centers both grids on zero however far the estimates sit from it. Runs that
keep each package's default grid therefore report ends taken from different
candidates.
:::

In [9]:
# The same 301 candidate effects for both packages, one every 0.0005 from -0.03 to 0.12.
grid = dict(grid_lb=-0.03, grid_ub=0.12, grid_points=301)

# R's helper returns the smallest and largest candidate that its test accepts at one value of M.
with ro.default_converter.context():
    ro.r("""
        smoothness_cs <- function(m, flag, bias = NULL, seed = 0) {
            args <- list(
                betahat = betahat, sigma = sigma, numPrePeriods = 4, numPostPeriods = 6, M = m,
                hybrid_flag = flag, gridPoints = 301, grid.lb = -0.03, grid.ub = 0.12, seed = seed
            )
            cs <- if (is.null(bias)) {
                do.call(HonestDiD::computeConditionalCS_DeltaSD, args)
            } else {
                do.call(HonestDiD::computeConditionalCS_DeltaSDB, c(args, biasDirection = bias))
            }
            range(cs$grid[cs$accept == 1])
        }
    """)

# Each method's name in honest_did, the hybrid flag that R's functions take for it, and the sign.
tests = [
    ("Conditional", "ARP", None),
    ("C-F", "FLCI", None),
    ("C-LF", "LF", None),
    ("C-F", "FLCI", "positive"),
]
tables = []
for method, flag, bias in tests:
    sign = {} if bias is None else {"bias_direction": bias}
    python_ci = did.honest_did(
        event_study,
        event_time=0,
        sensitivity_type="smoothness",
        m_vec=m_vec,
        method=method,
        **sign,
        **grid,
    ).robust_ci
    with ro.default_converter.context():
        ro.globalenv["flag"] = flag
        ro.globalenv["bias"] = ro.NULL if bias is None else bias
        r_bounds = np.asarray(ro.r("t(sapply(m_vec, smoothness_cs, flag = flag, bias = bias))"))
    labels = python_ci.select("method", "delta", "m").to_pandas()
    tables.append(side_by_side(labels, python_ci.select("lb", "ub").to_numpy(), r_bounds))
test_comparison = pd.concat(tables, ignore_index=True)
test_comparison

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
R callback write-console: In addition:   
R callback write-console: Warning messages:
  
R callback write-console: 1:   
R callback write-console: In base::sqrt(base::t(gamma_tilde) %*% sigma %*% gamma_tilde) :  
R callback write-console: 
   
R callback write-console:  NaNs produced
  
R callback write-console: 2:   
R callback write-console: In base::sqrt(base::t(gamma_tilde) %*% sigma %*% gamma_tilde) :  
R callback write-console: 
   
R callback write-console:  NaNs produced
  
R callback write-console: 3:   
R callback write-console: In base::sqrt(base::t(gamma_tilde) %*% sigma %*% gamma_tilde) :  
R callback write-console: 
   
R callback write-console:  NaNs produced
  
R callback write-console: In addition:   
R callback write-console: Warning messages:
  
R callback write-console: 1:   
R callback write-console: In base::sqrt(base::t(gamma_tilde) %*% sigma %*% gamma_tilde) :  
R ca

,method,delta,m,ModernDiD lb,R lb,lb difference,ModernDiD ub,R ub,ub difference
0,Conditional,DeltaSD,0.000,0.0250,0.0250,0.0000,0.0555,0.0555,0.0000
1,Conditional,DeltaSD,0.005,0.0210,0.0210,0.0000,0.0595,0.0595,0.0000
2,Conditional,DeltaSD,0.010,0.0170,0.0170,0.0000,0.0640,0.0640,0.0000
3,Conditional,DeltaSD,0.015,0.0125,0.0125,0.0000,0.0685,0.0685,0.0000
4,Conditional,DeltaSD,0.020,0.0075,0.0075,0.0000,0.0730,0.0730,0.0000
5,Conditional,DeltaSD,0.025,0.0025,0.0025,0.0000,0.0780,0.0780,0.0000
6,Conditional,DeltaSD,0.030,-0.0025,-0.0025,0.0000,0.0830,0.0830,0.0000
7,C-F,DeltaSD,0.000,0.0265,0.0265,0.0000,0.0555,0.0555,0.0000
8,C-F,DeltaSD,0.005,0.0215,0.0215,0.0000,0.0595,0.0595,0.0000
9,C-F,DeltaSD,0.010,0.0170,0.0170,0.0000,0.0635,0.0635,0.0000


The conditional test and the conditional FLCI hybrid accept the same candidates
in both packages at every $M$, in the DeltaSD rows without a sign and in the
DeltaSDPB rows with a positive bias. Although the hybrid's first stage uses
R's simulated quantile too, at the 99.5 percent level, the small shift moves no
candidate across the edge of a confidence set here. In the least favorable
hybrid, ModernDiD's interval is one step narrower at both ends up to
$M = 0.02$. At 0.025 and 0.03 only its upper end sits one step lower.

(r-comparisons-honest-seeds)=

### Seeds and the least favorable hybrid

Before it runs the conditional test, the least favorable hybrid rejects any
candidate whose test statistic exceeds a first-stage critical value. Each
package simulates that critical value as the 99.5th percentile of the statistic
over 1,000 simulated normal draws. Since ModernDiD draws with NumPy's generator
and `HonestDiD` with R's, the two critical values differ by simulation noise.
The next cell repeats this hybrid in each package under seeds 0 to 9 and lists
the range of each end.

In [10]:
# Run the least favorable hybrid under ten seeds in each package and keep the range of each end.
seeds = list(range(10))
python_runs = pl.concat(
    [
        did.honest_did(
            event_study,
            event_time=0,
            sensitivity_type="smoothness",
            m_vec=m_vec,
            method="C-LF",
            seed=seed,
            **grid,
        ).robust_ci.select("m", "lb", "ub")
        for seed in seeds
    ]
).with_columns(package=pl.lit("ModernDiD"))
with ro.default_converter.context():
    ro.globalenv["seeds"] = ro.IntVector(seeds)
    r_runs = np.asarray(
        ro.r("""
            do.call(rbind, lapply(seeds, function(seed) {
                t(sapply(m_vec, function(m) c(m, smoothness_cs(m, flag = "LF", seed = seed))))
            }))
        """)
    )
r_runs = pl.DataFrame(r_runs, schema=["m", "lb", "ub"], orient="row")

seed_ranges = (
    pl.concat([python_runs, r_runs.with_columns(package=pl.lit("R"))])
    .group_by("m", "package")
    .agg(
        pl.col("lb").min().alias("lowest lb"),
        pl.col("lb").max().alias("highest lb"),
        pl.col("ub").min().alias("lowest ub"),
        pl.col("ub").max().alias("highest ub"),
    )
    .sort("m", "package")
)
seed_ranges.to_pandas()

R callback write-console: In addition:   
R callback write-console: There were 30 warnings (use warnings() to see them)  
R callback write-console: 
  


,m,package,lowest lb,highest lb,lowest ub,highest ub
0,0.000,ModernDiD,0.0250,0.0255,0.0550,0.0560
1,0.000,R,0.0250,0.0250,0.0555,0.0555
2,0.005,ModernDiD,0.0210,0.0215,0.0590,0.0595
3,0.005,R,0.0210,0.0210,0.0595,0.0595
4,0.010,ModernDiD,0.0165,0.0170,0.0635,0.0640
5,0.010,R,0.0165,0.0165,0.0640,0.0640
6,0.015,ModernDiD,0.0120,0.0125,0.0680,0.0685
7,0.015,R,0.0120,0.0120,0.0685,0.0685
8,0.020,ModernDiD,0.0070,0.0075,0.0730,0.0735
9,0.020,R,0.0070,0.0070,0.0735,0.0735


Across the ten seeds, ModernDiD's upper end at $M = 0$ moves by two steps of
the grid and every other end by one. R's ends lie within ModernDiD's range at
every value of $M$. The one-step gaps in the least favorable hybrid are
therefore the noise of a simulation from 1,000 draws rather than a difference
between the tests.

:::{admonition} The seed argument in HonestDiD
:class: note

In the R rows, every seed gives the same two ends of each interval. With more
than one effect after treatment, `HonestDiD` 0.2.6 simulates the least
favorable critical value in `.ARP_computeCI`. Since that function passes no
seed to the simulation
([lines 737 to 740 of `R/arp-nuisance.R`](https://github.com/cran/HonestDiD/blob/0.2.6/R/arp-nuisance.R#L737-L740)),
the draws always use the default seed of 0 whatever `seed` you pass.
:::

## Relative magnitudes

Under relative magnitudes, both packages offer two methods, the conditional
test and the default least favorable hybrid. Neither offers fixed-length
intervals for this restriction, because such an interval would have to be
infinitely long. Since `createSensitivityResults_relativeMagnitudes` accepts a
grid, both packages test the same 301 candidates for each $\bar{M}$ from 0.5 to
3, the example's grid.

In [11]:
# Both packages test the example's relative magnitudes bounds on the shared candidates.
m_bar_vec = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
with ro.default_converter.context():
    ro.globalenv["m_bar_vec"] = ro.FloatVector(m_bar_vec)

tables = []
for method in ("C-LF", "Conditional"):
    python_ci = did.honest_did(
        event_study,
        event_time=0,
        sensitivity_type="relative_magnitude",
        m_bar_vec=m_bar_vec,
        method=method,
        **grid,
    ).robust_ci
    with ro.default_converter.context():
        ro.globalenv["method"] = method
        r_bounds = np.asarray(
            ro.r("""
                r_rm <- HonestDiD::createSensitivityResults_relativeMagnitudes(
                    betahat, sigma, numPrePeriods = 4, numPostPeriods = 6, method = method,
                    Mbarvec = m_bar_vec, gridPoints = 301, grid.lb = -0.03, grid.ub = 0.12
                )
                cbind(r_rm$lb, r_rm$ub)
            """)
        )
    labels = python_ci.select("method", "delta", "Mbar").to_pandas()
    tables.append(side_by_side(labels, python_ci.select("lb", "ub").to_numpy(), r_bounds))
rm_comparison = pd.concat(tables, ignore_index=True)
rm_comparison

,method,delta,Mbar,ModernDiD lb,R lb,lb difference,ModernDiD ub,R ub,ub difference
0,C-LF,DeltaRM,0.5,0.0300,0.0300,0.0000,0.0590,0.0590,0.0000
1,C-LF,DeltaRM,1.0,0.0235,0.0235,0.0000,0.0635,0.0635,0.0000
2,C-LF,DeltaRM,1.5,0.0160,0.0160,0.0000,0.0705,0.0705,0.0000
3,C-LF,DeltaRM,2.0,0.0085,0.0085,0.0000,0.0780,0.0780,0.0000
4,C-LF,DeltaRM,2.5,0.0005,0.0005,0.0000,0.0860,0.0855,0.0005
5,C-LF,DeltaRM,3.0,-0.0075,-0.0070,-0.0005,0.0935,0.0935,0.0000
6,Conditional,DeltaRM,0.5,0.0300,0.0300,0.0000,0.0585,0.0585,0.0000
7,Conditional,DeltaRM,1.0,0.0240,0.0240,0.0000,0.0635,0.0635,0.0000
8,Conditional,DeltaRM,1.5,0.0165,0.0165,0.0000,0.0705,0.0705,0.0000
9,Conditional,DeltaRM,2.0,0.0090,0.0090,0.0000,0.0775,0.0775,0.0000


The conditional test accepts the same candidates in both packages at every
$\bar{M}$. In the least favorable hybrid, ModernDiD's interval reaches one step
further at two ends. Its upper end sits one step higher at $\bar{M} = 2.5$ and
its lower end one step lower at $\bar{M} = 3$. Because R ignores its seed here,
the next cell reruns only ModernDiD at those two bounds over the same ten seeds.

In [12]:
# ModernDiD's least favorable hybrid at the two bounds where the packages differ.
rm_seed_runs = pl.concat(
    [
        did.honest_did(
            event_study,
            event_time=0,
            sensitivity_type="relative_magnitude",
            m_bar_vec=[2.5, 3.0],
            method="C-LF",
            seed=seed,
            **grid,
        ).robust_ci.select("Mbar", "lb", "ub")
        for seed in seeds
    ]
)
rm_seed_ranges = (
    rm_seed_runs.group_by("Mbar")
    .agg(
        pl.col("lb").min().alias("lowest lb"),
        pl.col("lb").max().alias("highest lb"),
        pl.col("ub").min().alias("lowest ub"),
        pl.col("ub").max().alias("highest ub"),
    )
    .sort("Mbar")
    .to_pandas()
)

# R's least favorable intervals at the same two bounds, from the table above.
r_least_favorable = rm_comparison[rm_comparison["method"] == "C-LF"].set_index("Mbar")
rm_seed_ranges["R lb"] = r_least_favorable.loc[rm_seed_ranges["Mbar"], "R lb"].to_numpy()
rm_seed_ranges["R ub"] = r_least_favorable.loc[rm_seed_ranges["Mbar"], "R ub"].to_numpy()
rm_seed_ranges

,Mbar,lowest lb,highest lb,lowest ub,highest ub,R lb,R ub
0,2.5,0.0005,0.0010,0.085,0.0860,0.0005,0.0855
1,3.0,-0.0075,-0.0065,0.093,0.0935,-0.0070,0.0935


Over the ten seeds, ModernDiD's upper end at $\bar{M} = 2.5$ runs from 0.0850
to 0.0860 and its lower end at $\bar{M} = 3$ from −0.0075 to −0.0065. R's ends
of 0.0855 and −0.0070 lie inside both ranges, as they did under smoothness.

## Shorter pre-treatment windows

The definitions of the restrictions fix how many placebo estimates each one
needs. The relative magnitudes restriction compares every yearly change from
the base year on with the largest yearly change before treatment. With a single
placebo estimate, that largest change is the step from event time −2 to the
base year. The linear-trend bound that `bound="deviation from linear trend"`
selects compares changes in slope instead. It needs two placebo estimates to
form one change in slope before treatment. Both packages therefore run relative
magnitudes with one placebo estimate and the linear-trend bound with two. The
next cell trims the event study to those minimums with `min_e` and runs the
conditional test under each restriction.

In [13]:
# Event studies with one and with two placebo estimates before the base year.
one_placebo = did.aggte(result, type="dynamic", min_e=-2, max_e=5)
two_placebos = did.aggte(result, type="dynamic", min_e=-3, max_e=5)

tables = []
for study, bound, num_pre in (
    (one_placebo, "deviation from parallel trends", 1),
    (two_placebos, "deviation from linear trend", 2),
):
    python_ci = did.honest_did(
        study,
        event_time=0,
        sensitivity_type="relative_magnitude",
        m_bar_vec=m_bar_vec,
        method="Conditional",
        bound=bound,
        **grid,
    ).robust_ci
    short_betahat, short_sigma = honest_inputs(study)
    with (ro.default_converter + numpy2ri.converter).context():
        ro.globalenv["short_betahat"] = short_betahat
        ro.globalenv["short_sigma"] = short_sigma
    with ro.default_converter.context():
        ro.globalenv["bound"] = bound
        ro.globalenv["num_pre"] = num_pre
        r_bounds = np.asarray(
            ro.r("""
                r_short <- HonestDiD::createSensitivityResults_relativeMagnitudes(
                    short_betahat, short_sigma, numPrePeriods = num_pre, numPostPeriods = 6,
                    bound = bound, method = "Conditional", Mbarvec = m_bar_vec,
                    gridPoints = 301, grid.lb = -0.03, grid.ub = 0.12
                )
                cbind(r_short$lb, r_short$ub)
            """)
        )
    labels = python_ci.select("delta", "Mbar").to_pandas()
    tables.append(side_by_side(labels, python_ci.select("lb", "ub").to_numpy(), r_bounds))
short_comparison = pd.concat(tables, ignore_index=True)
short_comparison

,delta,Mbar,ModernDiD lb,R lb,lb difference,ModernDiD ub,R ub,ub difference
0,DeltaRM,0.5,0.0310,0.0310,0.0,0.0585,0.0585,0.0
1,DeltaRM,1.0,0.0275,0.0275,0.0,0.0610,0.0610,0.0
2,DeltaRM,1.5,0.0225,0.0225,0.0,0.0650,0.0650,0.0
3,DeltaRM,2.0,0.0175,0.0175,0.0,0.0695,0.0695,0.0
4,DeltaRM,2.5,0.0120,0.0120,0.0,0.0745,0.0745,0.0
5,DeltaRM,3.0,0.0065,0.0065,0.0,0.0800,0.0800,0.0
6,DeltaSDRM,0.5,0.0205,0.0205,0.0,0.0570,0.0570,0.0
7,DeltaSDRM,1.0,0.0150,0.0150,0.0,0.0595,0.0595,0.0
8,DeltaSDRM,1.5,0.0080,0.0080,0.0,0.0645,0.0645,0.0
9,DeltaSDRM,2.0,0.0005,0.0005,0.0,0.0715,0.0715,0.0


Both restrictions give the same intervals in the two packages at every
$\bar{M}$.

(r-comparisons-honest-lower-bound)=

### The lower bound on M

The smoothness analysis has a diagnostic that needs two placebo estimates as
well, enough for one change in slope. Its lower bound on $M$ is the smallest $M$
that the placebo estimates don't reject. Both ModernDiD's
`compute_delta_sd_lowerbound_m` and R's `DeltaSD_lowerBound_Mpre` check for that
minimum before they compute anything. The next cell computes the bound in each
package with two placebo estimates and with three.

In [14]:
# The lower bound on M from two and from three placebo estimates.
three_placebos = did.aggte(result, type="dynamic", min_e=-4, max_e=5)
rows = []
for study, num_pre in ((two_placebos, 2), (three_placebos, 3)):
    lower_betahat, lower_sigma = honest_inputs(study)
    with (ro.default_converter + numpy2ri.converter).context():
        ro.globalenv["lower_betahat"] = lower_betahat
        ro.globalenv["lower_sigma"] = lower_sigma
    with ro.default_converter.context():
        ro.globalenv["num_pre"] = num_pre
        r_lower = ro.r("""
            tryCatch(
                format(HonestDiD::DeltaSD_lowerBound_Mpre(
                    lower_betahat, lower_sigma, numPrePeriods = num_pre
                )),
                error = function(e) paste("error:", conditionMessage(e))
            )
        """)[0]
    python_lower = did.compute_delta_sd_lowerbound_m(
        lower_betahat, lower_sigma, num_pre_periods=num_pre
    )
    rows.append({"placebo estimates": num_pre, "ModernDiD": python_lower, "R": r_lower})
pd.DataFrame(rows)

,placebo estimates,ModernDiD,R
0,2,0.0,error: subscript out of bounds
1,3,0.0,0


With two placebo estimates, ModernDiD returns a lower bound of 0, since the
three points from event time −3 to the base year are consistent with a straight
line. R stops with an error instead, although its own check allows two. In
`HonestDiD` 0.2.6,
[line 48 of `R/ublbM_functions.R`](https://github.com/cran/HonestDiD/blob/0.2.6/R/ublbM_functions.R#L48)
loops over `1:(numPrePeriods-2)` to fill in the remaining changes in slope.
When `numPrePeriods` is 2, that sequence counts down from 1 to 0 rather than
coming out empty. Its first pass then writes three coefficients into a row of a
matrix that has only two columns. Both packages return the same lower bound of
0 from three placebo estimates.

## Tolerances and assertions

The cell below turns each comparison into an assertion that a later rerun must
pass. The original interval and every interval from the conditional test or the
conditional FLCI hybrid must match to 1e-12. The fixed-length intervals may
differ by 1e-4 with R's simulated quantile and by 1e-6 with the exact one. That
second tolerance leaves room above the 5.4e-7 shown earlier because R's solvers
can stop at other weights when you rerun the page. The least favorable hybrid
may differ by one step of the shared grid. The table lists the largest gap in
each comparison next to its tolerance.

In [15]:
step = (grid["grid_ub"] - grid["grid_lb"]) / (grid["grid_points"] - 1)
smoothness_lf = test_comparison["method"] == "C-LF"
rm_lf = rm_comparison["method"] == "C-LF"
checks = [
    ("original interval", flci_comparison.iloc[:1], 1e-12),
    ("FLCI, simulated quantile in R", flci_comparison.iloc[1:], 1e-4),
    ("FLCI, exact quantile in R", flci_exact_comparison, 1e-6),
    ("Conditional and C-F, smoothness", test_comparison[~smoothness_lf], 1e-12),
    ("C-LF, smoothness", test_comparison[smoothness_lf], step + 1e-12),
    ("Conditional, relative magnitudes", rm_comparison[~rm_lf], 1e-12),
    ("C-LF, relative magnitudes", rm_comparison[rm_lf], step + 1e-12),
    ("Conditional, shorter windows", short_comparison, 1e-12),
]

rows = []
for label, table, tolerance in checks:
    np.testing.assert_allclose(table["ModernDiD lb"], table["R lb"], rtol=0, atol=tolerance)
    np.testing.assert_allclose(table["ModernDiD ub"], table["R ub"], rtol=0, atol=tolerance)
    rows.append(
        {
            "comparison": label,
            "largest lb difference": table["lb difference"].abs().max(),
            "largest ub difference": table["ub difference"].abs().max(),
            "tolerance": tolerance,
        }
    )
pd.DataFrame(rows)

,comparison,largest lb difference,largest ub difference,tolerance
0,original interval,6.93889390e-18,6.93889390e-18,1.00000000e-12
1,"FLCI, simulated quantile in R",5.57386314e-05,7.51653402e-05,1.00000000e-04
2,"FLCI, exact quantile in R",5.41509480e-07,5.41867282e-07,1.00000000e-06
3,"Conditional and C-F, smoothness",0.00000000e+00,0.00000000e+00,1.00000000e-12
4,"C-LF, smoothness",5.00000000e-04,5.00000000e-04,5.00000001e-04
5,"Conditional, relative magnitudes",0.00000000e+00,0.00000000e+00,1.00000000e-12
6,"C-LF, relative magnitudes",5.00000000e-04,5.00000000e-04,5.00000001e-04
7,"Conditional, shorter windows",0.00000000e+00,0.00000000e+00,1.00000000e-12


## Versions used

The output above came from the versions of Python, R, `rpy2`, and the two
packages listed below.

In [16]:
from importlib.metadata import version
from platform import python_version

with ro.default_converter.context():
    r_version = str(ro.r("R.version.string")[0])
    honestdid_version = str(ro.r('as.character(packageVersion("HonestDiD"))')[0])
pd.DataFrame(
    {
        "component": ["Python", "ModernDiD", "rpy2", "R", "HonestDiD"],
        "version": [
            python_version(),
            version("moderndid"),
            version("rpy2"),
            r_version,
            honestdid_version,
        ],
    }
)

,component,version
0,Python,3.12.10
1,ModernDiD,0.2.0
2,rpy2,3.6.4
3,R,R version 4.3.3 (2024-02-29)
4,HonestDiD,0.2.6
